In [1]:
!git clone https://github.com/manhar1087/amazon-ml-challenge.git /kaggle/working/amazon_ml_challenge

Cloning into '/kaggle/working/amazon_ml_challenge'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 115 (delta 11), reused 115 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 140.41 KiB | 2.60 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [2]:
%cd /kaggle/working/amazon_ml_challenge

/kaggle/working/amazon_ml_challenge


In [3]:
!git status

On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean


In [4]:
!ls

audit_final.py			    patch.py
benchmark_char_tfidf.py		    prototype_external_df.py
benchmark_char_tfidf_small.py	    prototype_global_topk.py
build_full_corpus.py		    prototype_streaming_input_df.py
check_dist.py			    prototype_streaming_tfidf_deterministic.py
deterministic_corpus_manifest.json  prototype_streaming_tfidf.py
diagnose_misses.py		    README.md
diagnose_v2_misses.py		    requirements.txt
Documentation_template.md	    retrieval_benchmark.py
eval_channels.py		    run_full_pipeline.py
eval_holdout_recall.py		    run_recall_comparison.py
evaluate_mixed_k.py		    src
evaluate_task03.py		    step123_test.py
evaluate_v2_comprehensive.py	    task03_final_manifest.json
evaluate_v3_final.py		    test_chunked_deterministic_merge.py
evaluate_v4_full.py		    tests
experiment_runner.py		    test_sparse.py
experiment_runner_v2.py		    test_tie_breaking.py
fix_and_run.py			    train_task04.py
fix.py				    utils
full_build_manifest.json	    verify_correctness_fix.py
generate_task03_hol

In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

input/
  datasets/
    manhar04/
      amazon-ml-challenge-2026-data/
        test/
          test_source2.tsv
          test_source3.tsv
          test_source1.tsv
        train/
          train_ground_truth.tsv
          train_source3.tsv
          train_source2.tsv
          train_source1.tsv


In [6]:
%cd /kaggle/working/amazon_ml_challenge
!pwd
!git status

/kaggle/working/amazon_ml_challenge
/kaggle/working/amazon_ml_challenge
On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean


In [7]:
!sed -n '1,240p' src/data/loader.py

import polars as pl
import os
from .normalization import add_normalized_columns

def load_and_convert_tsv(tsv_path: str, parquet_path: str, force: bool = False) -> pl.LazyFrame:
    """
    Converts a TSV file to Parquet format for fast subsequent loads.
    Applies normalization pipeline during conversion.
    Returns a LazyFrame pointing to the Parquet file.
    """
    os.makedirs(os.path.dirname(parquet_path), exist_ok=True)
    
    if force or not os.path.exists(parquet_path):
        print(f"Converting {tsv_path} to {parquet_path}...")
        # Read raw TSV
        lf = pl.scan_csv(
            tsv_path, 
            separator="\t",
            infer_schema_length=10000,
            null_values=["", "NaN", "null"],
            missing_utf8_is_empty_string=False
        )
        
        # Apply normalization
        lf = add_normalized_columns(lf)
        
        # Write to Parquet
        # Collect executes the graph
        df = lf.collect(streaming=True)
        df.write_p

In [8]:
%cd /kaggle/working/amazon_ml_challenge

/kaggle/working/amazon_ml_challenge


In [9]:
!ln -s /kaggle/input/datasets/manhar04/amazon-ml-challenge-2026-data dataset

In [10]:
!ls -lah dataset
!ls dataset/train
!ls dataset/test

lrwxrwxrwx 1 root root 61 Sep 25 17:47 dataset -> /kaggle/input/datasets/manhar04/amazon-ml-challenge-2026-data
train_ground_truth.tsv	train_source1.tsv  train_source2.tsv  train_source3.tsv
test_source1.tsv  test_source2.tsv  test_source3.tsv


In [11]:
import polars as pl

df = pl.scan_csv(
    "dataset/train/train_source1.tsv",
    separator="\t",
    infer_schema_length=1000
)

print(df.collect_schema())

Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})


In [13]:
!pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.8/865.8 kB 12.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.3 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-polars-cu12 26.2.1 requires polars<1.36,>=1.30, but you have polars 1.44.2 which is incompatible.


In [15]:
!pip install rapidfuzz -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.9 MB/s eta 0:00:00a 0:00:01


In [16]:
import polars
import pyarrow
import sklearn
import lightgbm
import rapidfuzz
import numpy
import torch

print("polars:", polars.__version__)
print("pyarrow:", pyarrow.__version__)
print("sklearn:", sklearn.__version__)
print("lightgbm:", lightgbm.__version__)
print("rapidfuzz:", rapidfuzz.__version__)
print("numpy:", numpy.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

polars: 1.35.2
pyarrow: 24.0.0
sklearn: 1.6.1
lightgbm: 4.6.0
rapidfuzz: 3.14.6
numpy: 2.0.2
CUDA available: True
GPU count: 2


In [17]:
from src.data.loader import load_all_data
from src.data.normalization import add_normalized_columns
from src.features.feature_engineering import build_features

print("Project imports: OK")

Project imports: OK


In [18]:
%cd /kaggle/working/amazon_ml_challenge

import os
import torch

print("=== ENVIRONMENT ===")
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

print("\n=== DATA ===")
base = "/kaggle/input/datasets/manhar04/amazon-ml-challenge-2026-data"
for split in ["train", "test"]:
    print(split, os.listdir(os.path.join(base, split)))

print("\n=== PROJECT ===")
print("train_task04.py:", os.path.exists("train_task04.py"))
print("generate_task03_holdout.py:", os.path.exists("generate_task03_holdout.py"))
print("requirements.txt:", os.path.exists("requirements.txt"))

print("\n=== EXISTING ARTIFACTS ===")
for p in [
    "work/task04/lgb_model.pkl",
    "work/task04/val_predictions.parquet",
    "work/candidates/task04_train_val_cands.parquet",
]:
    print(p, "→", os.path.exists(p))

/kaggle/working/amazon_ml_challenge
=== ENVIRONMENT ===
CUDA: True
GPU count: 2

=== DATA ===
train ['train_ground_truth.tsv', 'train_source3.tsv', 'train_source2.tsv', 'train_source1.tsv']
test ['test_source2.tsv', 'test_source3.tsv', 'test_source1.tsv']

=== PROJECT ===
train_task04.py: True
generate_task03_holdout.py: True
requirements.txt: True

=== EXISTING ARTIFACTS ===
work/task04/lgb_model.pkl → False
work/task04/val_predictions.parquet → False
work/candidates/task04_train_val_cands.parquet → False


In [19]:
%cd /kaggle/working/amazon_ml_challenge
!python -u train_task04.py

/kaggle/working/amazon_ml_challenge
Loading data...
Converting dataset/train/train_source1.tsv to work/parquet/train/source1.parquet...
Finished writing work/parquet/train/source1.parquet
Converting dataset/train/train_source2.tsv to work/parquet/train/source2.parquet...
Finished writing work/parquet/train/source2.parquet
Converting dataset/train/train_source3.tsv to work/parquet/train/source3.parquet...
Finished writing work/parquet/train/source3.parquet
Converting dataset/test/test_source1.tsv to work/parquet/test/source1.parquet...
Finished writing work/parquet/test/source1.parquet
Converting dataset/test/test_source2.tsv to work/parquet/test/source2.parquet...
Finished writing work/parquet/test/source2.parquet
Converting dataset/test/test_source3.tsv to work/parquet/test/source3.parquet...
Finished writing work/parquet/test/source3.parquet
Splitting dataset...
Generating canonical candidates for Train/Val sets...
Loading corpus IDs...
[tfidf_name_k50] Loading precomputed vocab and 

In [20]:
%cd /kaggle/working/amazon_ml_challenge
!python -u build_full_corpus.py

/kaggle/working/amazon_ml_challenge
FULL 10.32M CORPUS BUILD - TASK 03
Source2 rows: 5034616
Source3 rows: 5285603
Total Corpus Rows: 10320219

--- Channel: name_word ---
  Building vocab & IDF for name_word...
  DF batched in 45.9s. Merging 104 files...
  Vocab (276917 terms) built in 58.8s. Peak RSS=328.6 MiB
  Building CSR chunks for name_word...
  Chunks built: 207. Total NNZ: 23835568. Time: 157.8s. Peak RSS: 337.3 MiB

--- Channel: address_word ---
  Building vocab & IDF for address_word...
  DF batched in 170.6s. Merging 104 files...
  Vocab (4953784 terms) built in 285.7s. Peak RSS=1377.2 MiB
  Building CSR chunks for address_word...
  Chunks built: 207. Total NNZ: 112660296. Time: 327.3s. Peak RSS: 1298.9 MiB

--- Channel: name_char ---
  Building vocab & IDF for name_char...
  DF batched in 187.2s. Merging 104 files...
  Vocab (281091 terms) built in 209.5s. Peak RSS=1306.6 MiB
  Building CSR chunks for name_char...
  Chunks built: 207. Total NNZ: 147453990. Time: 366.5s. Pea

In [21]:
!ls -lah retrieval_setup/name_word
!ls -lah retrieval_setup/address_word
!ls -lah retrieval_setup/name_char

total 6.3M
drwxr-xr-x 2 root root 4.0K Sep 25 17:57 .
drwxr-xr-x 5 root root 4.0K Sep 25 18:14 ..
-rw-r--r-- 1 root root 1.1M Sep 25 17:57 idf.npy
-rw-r--r-- 1 root root 5.3M Sep 25 17:57 vocab.json
total 133M
drwxr-xr-x 2 root root 4.0K Sep 25 18:05 .
drwxr-xr-x 5 root root 4.0K Sep 25 18:14 ..
-rw-r--r-- 1 root root  19M Sep 25 18:05 idf.npy
-rw-r--r-- 1 root root 114M Sep 25 18:05 vocab.json
total 5.3M
drwxr-xr-x 2 root root 4.0K Sep 25 18:14 .
drwxr-xr-x 5 root root 4.0K Sep 25 18:14 ..
-rw-r--r-- 1 root root 1.1M Sep 25 18:14 idf.npy
-rw-r--r-- 1 root root 4.2M Sep 25 18:14 vocab.json


In [22]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files)

/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/manhar04 []
/kaggle/input/datasets/manhar04/amazon-ml-task04-artifacts []
/kaggle/input/datasets/manhar04/amazon-ml-task04-artifacts/candidates ['task04_train_val_cands.parquet']
/kaggle/input/datasets/manhar04/amazon-ml-task04-artifacts/task04 ['train_val_ids.json', 'experiment_config.json', 'val_predictions.parquet', 'lgb_model.pkl', 'feature_schema.json']
/kaggle/input/datasets/manhar04/amazon-ml-challenge-2026-data []
/kaggle/input/datasets/manhar04/amazon-ml-challenge-2026-data/test ['test_source2.tsv', 'test_source3.tsv', 'test_source1.tsv']
/kaggle/input/datasets/manhar04/amazon-ml-challenge-2026-data/train ['train_ground_truth.tsv', 'train_source3.tsv', 'train_source2.tsv', 'train_source1.tsv']


In [23]:
from pathlib import Path
import shutil

src = Path("/kaggle/input/datasets/manhar04/amazon-ml-task04-artifacts")
dst = Path("/kaggle/working/amazon_ml_challenge/work")

(dst / "candidates").mkdir(parents=True, exist_ok=True)
(dst / "task04").mkdir(parents=True, exist_ok=True)

shutil.copy2(
    src / "candidates/task04_train_val_cands.parquet",
    dst / "candidates/task04_train_val_cands.parquet"
)

for f in (src / "task04").iterdir():
    shutil.copy2(f, dst / "task04" / f.name)

print("Task 04 artifacts copied successfully.\n")

for p in [
    dst / "candidates/task04_train_val_cands.parquet",
    dst / "task04/lgb_model.pkl",
    dst / "task04/feature_schema.json",
    dst / "task04/val_predictions.parquet",
    dst / "task04/train_val_ids.json",
    dst / "task04/experiment_config.json",
]:
    print(f"{p.relative_to('/kaggle/working/amazon_ml_challenge')} : {p.stat().st_size:,} bytes")

Task 04 artifacts copied successfully.

work/candidates/task04_train_val_cands.parquet : 15,629,058 bytes
work/task04/lgb_model.pkl : 164,627 bytes
work/task04/feature_schema.json : 1,654 bytes
work/task04/val_predictions.parquet : 8,752,106 bytes
work/task04/train_val_ids.json : 317,773 bytes
work/task04/experiment_config.json : 488 bytes


In [24]:
%cd /kaggle/working/amazon_ml_challenge
!ls -lh work/candidates/task04_train_val_cands.parquet
!ls -lh work/task04

/kaggle/working/amazon_ml_challenge
-rw-r--r-- 1 root root 15M Sep 25 18:25 work/candidates/task04_train_val_cands.parquet
total 8.9M
-rw-r--r-- 1 root root  488 Sep 25 18:25 experiment_config.json
-rw-r--r-- 1 root root 1.7K Sep 25 18:25 feature_schema.json
-rw-r--r-- 1 root root 161K Sep 25 18:25 lgb_model.pkl
-rw-r--r-- 1 root root 311K Sep 25 18:25 train_val_ids.json
-rw-r--r-- 1 root root 8.4M Sep 25 18:25 val_predictions.parquet


In [25]:
%cd /kaggle/working/amazon_ml_challenge
!python -u train_task04.py

/kaggle/working/amazon_ml_challenge
Loading data...
Splitting dataset...
Loading existing candidates from work/candidates/task04_train_val_cands.parquet...
Validation Positive Pair Coverage after Task 03 Blocking: 0.9401
Labeling train candidates...
Train Positives: 32584
Train Negatives: 168044
Building Training Features...
Training features built in 6.09s
Building Validation Features...
Validation features built in 11.85s
Training LightGBM Model...
Training until validation scores don't improve for 20 rounds
[20]	training's auc: 0.99759	training's average_precision: 0.988244	valid_1's auc: 0.993659	valid_1's average_precision: 0.900749
[40]	training's auc: 0.998478	training's average_precision: 0.99192	valid_1's auc: 0.994562	valid_1's average_precision: 0.921527
[60]	training's auc: 0.998804	training's average_precision: 0.993607	valid_1's auc: 0.994236	valid_1's average_precision: 0.927411
Early stopping, best iteration is:
[43]	training's auc: 0.998533	training's average_precision

In [26]:
!sed -n '1,360p' src/modeling/task05.py

import time
import os
import polars as pl
import numpy as np
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
import pickle

from src.data.loader import load_all_data
from src.evaluation.splits import get_grouped_split
from src.features.feature_engineering import build_features
from src.evaluation.candidate_recall import evaluate_candidates

def run_task05():
    print("Loading data...")
    frames = load_all_data()
    s1 = frames["train_source1"]
    s23 = pl.concat([frames["train_source2"], frames["train_source3"]], how="diagonal")
    gt = frames["train_ground_truth"].collect()
    
    gt_dict = {row["source1_entity_id"]: set(row["matched_entity_ids"].split(",")) if row["matched_entity_ids"] else set() for row in gt.iter_rows(named=True)}
    
    s1_ids_all = gt["source1_entity_id"].to_list()
    train_s1_ids, val_s1_ids = get_grouped_split(s1_ids_all)
    
    # 1. Reproduce Task 04 split to find untouched S1s for calibration
    np.random.seed(42)
    trai

In [27]:
import polars as pl
import json

p = "work/task04/val_predictions.parquet"

df = pl.read_parquet(p)

print("Rows:", df.height)
print("Columns:", df.columns)
print(df.head())

print("\nExperiment config:")
print(json.load(open("work/task04/experiment_config.json")))

Rows: 850622
Columns: ['source1_entity_id', 'candidate_entity_id', 'label', 'pred']
shape: (5, 4)
┌───────────────────┬─────────────────────┬───────┬──────────┐
│ source1_entity_id ┆ candidate_entity_id ┆ label ┆ pred     │
│ ---               ┆ ---                 ┆ ---   ┆ ---      │
│ str               ┆ str                 ┆ f32   ┆ f64      │
╞═══════════════════╪═════════════════════╪═══════╪══════════╡
│ S1-161370177      ┆ S2-94892420         ┆ 0.0   ┆ 0.007341 │
│ S1-659489269      ┆ S2-878969226        ┆ 0.0   ┆ 0.002292 │
│ S1-155939771      ┆ S2-177142067        ┆ 0.0   ┆ 0.002292 │
│ S1-930667369      ┆ S3-297744193        ┆ 0.0   ┆ 0.002292 │
│ S1-268529601      ┆ S2-980113965        ┆ 0.0   ┆ 0.002292 │
└───────────────────┴─────────────────────┴───────┴──────────┘

Experiment config:
{'roc_auc': 0.9946633684586661, 'pr_auc': 0.9231030597259606, 'pair_precision': 0.7478324244175819, 'pair_recall': 0.8600786172611133, 'entity_macro_f05': 0.7291424204736537, 'train_s1_coun

In [28]:
import json
import numpy as np
import polars as pl

# -----------------------------
# Load Task 04 validation data
# -----------------------------
preds = pl.read_parquet("work/task04/val_predictions.parquet")
gt = pl.read_csv(
    "dataset/train/train_ground_truth.tsv",
    separator="\t"
)

print("Prediction rows:", preds.height)

# These are exactly the S1s represented in the Task 04 validation predictions
val_s1_ids = preds["source1_entity_id"].unique().to_list()
print("Validation S1 count:", len(val_s1_ids))

# -----------------------------
# Build true match counts
# -----------------------------
gt_dict = {}

for row in gt.iter_rows(named=True):
    s1 = row["source1_entity_id"]
    raw = row["matched_entity_ids"]

    if raw is None or str(raw).strip() == "":
        gt_dict[s1] = set()
    else:
        gt_dict[s1] = {
            x.strip()
            for x in str(raw).split(",")
            if x.strip()
        }

s1_to_idx = {s1: i for i, s1 in enumerate(val_s1_ids)}

true_counts = np.array(
    [len(gt_dict.get(s1, set())) for s1 in val_s1_ids],
    dtype=np.int32
)

# -----------------------------
# Convert prediction data
# -----------------------------
s1_idx = np.array(
    [s1_to_idx[x] for x in preds["source1_entity_id"].to_list()],
    dtype=np.int32
)

scores = preds["pred"].to_numpy()
labels = preds["label"].to_numpy()

# -----------------------------
# Exact entity-level Macro F0.5
# -----------------------------
def macro_f05(threshold):
    selected = scores >= threshold

    pred_counts = np.bincount(
        s1_idx[selected],
        minlength=len(val_s1_ids)
    )

    tp = np.bincount(
        s1_idx[selected & (labels > 0.5)],
        minlength=len(val_s1_ids)
    )

    denom = 0.25 * true_counts + pred_counts

    f = np.divide(
        1.25 * tp,
        denom,
        out=np.zeros(len(val_s1_ids), dtype=np.float64),
        where=denom > 0
    )

    # True empty + predicted empty = perfect
    f[(true_counts == 0) & (pred_counts == 0)] = 1.0

    return float(f.mean())


# -----------------------------
# Coarse sweep
# -----------------------------
print("\n=== COARSE THRESHOLD SWEEP ===")

thresholds = np.arange(0.01, 1.00, 0.01)

results = []

for th in thresholds:
    score = macro_f05(th)
    results.append((float(th), score))
    print(f"{th:.2f} -> {score:.6f}")

best_th, best_score = max(results, key=lambda x: x[1])

print("\nBest coarse threshold:", best_th)
print("Best coarse Macro F0.5:", best_score)


# -----------------------------
# Fine sweep around best
# -----------------------------
print("\n=== FINE SWEEP ===")

lo = max(0.001, best_th - 0.05)
hi = min(0.999, best_th + 0.05)

fine_thresholds = np.arange(lo, hi + 0.0005, 0.001)

fine_results = []

for th in fine_thresholds:
    score = macro_f05(th)
    fine_results.append((float(th), score))

best_th, best_score = max(fine_results, key=lambda x: x[1])

print("\nBEST THRESHOLD:", round(best_th, 4))
print("BEST MACRO F0.5:", round(best_score, 6))


# -----------------------------
# Breakdown at best threshold
# -----------------------------
selected = scores >= best_th

pred_counts = np.bincount(
    s1_idx[selected],
    minlength=len(val_s1_ids)
)

tp = np.bincount(
    s1_idx[selected & (labels > 0.5)],
    minlength=len(val_s1_ids)
)

denom = 0.25 * true_counts + pred_counts

f = np.divide(
    1.25 * tp,
    denom,
    out=np.zeros(len(val_s1_ids), dtype=np.float64),
    where=denom > 0
)

f[(true_counts == 0) & (pred_counts == 0)] = 1.0

zero_mask = true_counts == 0
singleton_mask = true_counts == 1
multi_mask = true_counts > 1

print("\n=== BREAKDOWN ===")
print("Zero-match S1s:", zero_mask.sum())
print("Singleton S1s:", singleton_mask.sum())
print("Multi-match S1s:", multi_mask.sum())

print("Zero-match F0.5:", round(float(f[zero_mask].mean()), 6))
print("Singleton F0.5:", round(float(f[singleton_mask].mean()), 6))
print("Multi-match F0.5:", round(float(f[multi_mask].mean()), 6))

# Save
result = {
    "best_threshold": best_th,
    "best_macro_f05": best_score,
    "validation_s1_count": len(val_s1_ids),
    "prediction_rows": preds.height
}

with open("work/task05_threshold_result.json", "w") as f_out:
    json.dump(result, f_out, indent=2)

print("\nSaved: work/task05_threshold_result.json")

Prediction rows: 850622
Validation S1 count: 10000

=== COARSE THRESHOLD SWEEP ===
0.01 -> 0.229212
0.02 -> 0.340890
0.03 -> 0.443311
0.04 -> 0.523078
0.05 -> 0.567293
0.06 -> 0.589904
0.07 -> 0.607473
0.08 -> 0.620296
0.09 -> 0.630430
0.10 -> 0.638803
0.11 -> 0.645526
0.12 -> 0.651539
0.13 -> 0.656398
0.14 -> 0.660445
0.15 -> 0.664730
0.16 -> 0.668936
0.17 -> 0.671934
0.18 -> 0.674989
0.19 -> 0.677331
0.20 -> 0.679901
0.21 -> 0.682275
0.22 -> 0.684762
0.23 -> 0.687137
0.24 -> 0.689094
0.25 -> 0.691319
0.26 -> 0.692966
0.27 -> 0.695062
0.28 -> 0.696913
0.29 -> 0.698549
0.30 -> 0.700332
0.31 -> 0.701820
0.32 -> 0.703250
0.33 -> 0.704734
0.34 -> 0.706491
0.35 -> 0.707916
0.36 -> 0.709290
0.37 -> 0.710652
0.38 -> 0.712006
0.39 -> 0.713748
0.40 -> 0.715170
0.41 -> 0.716480
0.42 -> 0.717854
0.43 -> 0.719321
0.44 -> 0.720827
0.45 -> 0.722267
0.46 -> 0.723760
0.47 -> 0.724996
0.48 -> 0.726415
0.49 -> 0.727750
0.50 -> 0.729142
0.51 -> 0.730891
0.52 -> 0.732230
0.53 -> 0.734185
0.54 -> 0.738282

In [29]:
import numpy as np
import polars as pl

preds = pl.read_parquet("work/task04/val_predictions.parquet")

# True labels already exist in the candidate set.
# Predict every true candidate: this is the "oracle matcher" ceiling
# for the current candidate generation stage.

s1_ids = preds["source1_entity_id"].unique().to_list()
s1_to_idx = {x: i for i, x in enumerate(s1_ids)}

idx = np.array([s1_to_idx[x] for x in preds["source1_entity_id"]])
y = preds["label"].to_numpy()

true_counts = np.bincount(
    idx[y > 0.5],
    minlength=len(s1_ids)
)

candidate_counts = np.bincount(
    idx,
    minlength=len(s1_ids)
)

# Oracle prediction = every candidate whose label is actually positive
tp = true_counts

denom = 0.25 * true_counts + tp
oracle_f = np.divide(
    1.25 * tp,
    denom,
    out=np.zeros(len(s1_ids)),
    where=denom > 0
)

oracle_f[(true_counts == 0) & (candidate_counts > 0)] = 0.0
oracle_f[(true_counts == 0) & (candidate_counts == 0)] = 1.0

print("=== CURRENT CANDIDATE-SET CEILING ===")
print("Oracle Macro F0.5:", oracle_f.mean())

# Candidate recall
candidate_recall = np.divide(
    tp,
    true_counts,
    out=np.ones(len(s1_ids), dtype=float),
    where=true_counts > 0
)

nonzero = true_counts > 0

print("Pair recall:", tp[nonzero].sum() / true_counts[nonzero].sum())
print("Mean per-S1 recall:", candidate_recall[nonzero].mean())
print("Worst per-S1 recall:", candidate_recall[nonzero].min())
print("S1s with incomplete candidate recall:", (candidate_recall[nonzero] < 1).sum())

=== CURRENT CANDIDATE-SET CEILING ===
Oracle Macro F0.5: 0.9377
Pair recall: 1.0
Mean per-S1 recall: 1.0
Worst per-S1 recall: 1.0
S1s with incomplete candidate recall: 0


In [32]:
import json
import numpy as np
import polars as pl

cands = pl.read_parquet(
    "work/candidates/task04_train_val_cands.parquet"
)

gt = pl.read_csv(
    "dataset/train/train_ground_truth.tsv",
    separator="\t"
)

split_info = json.load(
    open("work/task04/train_val_ids.json")
)

val_ids = split_info["val_ids"]

gt_dict = {}

for row in gt.iter_rows(named=True):
    raw = row["matched_entity_ids"]

    if raw is None or str(raw).strip() == "":
        gt_dict[row["source1_entity_id"]] = set()
    else:
        gt_dict[row["source1_entity_id"]] = {
            x.strip()
            for x in str(raw).split(",")
            if x.strip()
        }

cand_dict = {}

val_cands = cands.filter(
    pl.col("source1_entity_id").is_in(val_ids)
)

for row in val_cands.iter_rows(named=True):
    cand_dict.setdefault(
        row["source1_entity_id"], set()
    ).add(row["candidate_entity_id"])

pair_tp = 0
pair_total = 0

per_s1_recall = []
oracle_f05 = []

zero_count = 0
zero_with_candidates = 0

for s1 in val_ids:

    true_set = gt_dict.get(s1, set())
    cand_set = cand_dict.get(s1, set())

    # Zero-match entity:
    # oracle predicts EMPTY, regardless of candidate presence.
    if not true_set:
        zero_count += 1
        if cand_set:
            zero_with_candidates += 1

        oracle_f05.append(1.0)
        continue

    recovered = len(true_set & cand_set)

    pair_tp += recovered
    pair_total += len(true_set)

    recall = recovered / len(true_set)
    per_s1_recall.append(recall)

    # Oracle prediction = every true match that was retrieved.
    pred_count = recovered

    if recovered == 0:
        oracle_f05.append(0.0)
        continue

    precision = 1.0
    recall = recovered / len(true_set)

    f05 = (
        1.25 * precision * recall
        / (0.25 * precision + recall)
    )

    oracle_f05.append(f05)

print("=== CORRECT CANDIDATE-LIMITED CEILING ===")
print("Pair recall:", pair_tp / pair_total)
print("Mean nonzero-S1 recall:", np.mean(per_s1_recall))
print("Worst nonzero-S1 recall:", np.min(per_s1_recall))
print(
    "Nonzero S1s with incomplete recall:",
    sum(r < 1.0 for r in per_s1_recall)
)

print("\nZero-match S1s:", zero_count)
print("Zero-match S1s with candidates:", zero_with_candidates)

print(
    "\nOracle Entity Macro F0.5:",
    np.mean(oracle_f05)
)

=== CORRECT CANDIDATE-LIMITED CEILING ===
Pair recall: 0.9400543384010637
Mean nonzero-S1 recall: 0.9409047345195564
Worst nonzero-S1 recall: 0.0
Nonzero S1s with incomplete recall: 1586

Zero-match S1s: 568
Zero-match S1s with candidates: 568

Oracle Entity Macro F0.5: 0.9792207951178927


In [33]:
import numpy as np
import polars as pl

df = pl.read_parquet("work/task04/val_predictions.parquet")

TH = 0.926

# prediction at current tuned threshold
df = df.with_columns(
    (pl.col("pred") >= TH).alias("selected")
)

# Per-S1 statistics
stats = (
    df.group_by("source1_entity_id")
      .agg([
          pl.col("label").sum().alias("true_retrieved"),
          pl.col("selected").sum().alias("predicted"),
          ((pl.col("label") == 1) & (pl.col("pred") >= TH))
              .sum()
              .alias("tp"),
          pl.len().alias("candidate_count"),
          pl.col("pred").max().alias("max_score"),
      ])
)

# Ground truth
gt = pl.read_csv(
    "dataset/train/train_ground_truth.tsv",
    separator="\t"
)

gt = gt.with_columns(
    pl.when(
        pl.col("matched_entity_ids").is_null() |
        (pl.col("matched_entity_ids") == "")
    )
    .then(0)
    .otherwise(
        pl.col("matched_entity_ids").str.split(",").list.len()
    )
    .alias("true_count")
)

stats = stats.join(
    gt.select([
        "source1_entity_id",
        "true_count"
    ]),
    on="source1_entity_id",
    how="left"
)

stats = stats.with_columns(
    (pl.col("predicted") - pl.col("true_count")).alias("count_error")
)

print("=== MATCHING DIAGNOSTIC ===")

print("\nPrediction count distribution:")
print(
    stats.group_by("predicted")
         .agg(pl.len().alias("s1_count"))
         .sort("predicted")
         .head(30)
)

print("\nTrue count distribution:")
print(
    stats.group_by("true_count")
         .agg(pl.len().alias("s1_count"))
         .sort("true_count")
)

print("\nWrong match-count S1s:")
print(
    stats.filter(
        pl.col("predicted") != pl.col("true_count")
    ).height
)

print("\nExact match-count S1s:")
print(
    stats.filter(
        pl.col("predicted") == pl.col("true_count")
    ).height
)

print("\nFalse-positive-heavy S1s:")
print(
    stats.filter(
        (pl.col("predicted") > pl.col("tp")) &
        (pl.col("predicted") > 0)
    ).height
)

print("\nMissed-true-match S1s:")
print(
    stats.filter(
        pl.col("tp") < pl.col("true_retrieved")
    ).height
)

print("\nScore quantiles:")
print(
    df.select(
        pl.col("pred").quantile(
            [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
        )
    )
)

print("\nPositive score quantiles:")
print(
    df.filter(pl.col("label") == 1)
      .select(
          pl.col("pred").quantile(
              [0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.9]
          )
      )
)

print("\nNegative score quantiles:")
print(
    df.filter(pl.col("label") == 0)
      .select(
          pl.col("pred").quantile(
              [0.9, 0.95, 0.99, 0.999]
          )
      )
)

=== MATCHING DIAGNOSTIC ===

Prediction count distribution:
shape: (11, 2)
┌───────────┬──────────┐
│ predicted ┆ s1_count │
│ ---       ┆ ---      │
│ u32       ┆ u32      │
╞═══════════╪══════════╡
│ 0         ┆ 766      │
│ 1         ┆ 1531     │
│ 2         ┆ 2340     │
│ 3         ┆ 2293     │
│ 4         ┆ 1630     │
│ …         ┆ …        │
│ 6         ┆ 357      │
│ 7         ┆ 139      │
│ 8         ┆ 30       │
│ 9         ┆ 9        │
│ 10        ┆ 1        │
└───────────┴──────────┘

True count distribution:
shape: (12, 2)
┌────────────┬──────────┐
│ true_count ┆ s1_count │
│ ---        ┆ ---      │
│ u32        ┆ u32      │
╞════════════╪══════════╡
│ 0          ┆ 568      │
│ 1          ┆ 530      │
│ 2          ┆ 1682     │
│ 3          ┆ 2382     │
│ 4          ┆ 2236     │
│ …          ┆ …        │
│ 7          ┆ 284      │
│ 8          ┆ 84       │
│ 9          ┆ 15       │
│ 10         ┆ 3        │
│ 11         ┆ 1        │
└────────────┴──────────┘

Wrong match-coun

ComputeError: could not extract number from any-value of dtype: 'List(Float64)'

In [34]:
# -----------------------------
# Score quantiles
# -----------------------------

def quantiles(series, qs):
    return {
        q: float(series.quantile(q))
        for q in qs
    }

qs_all = [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
qs_pos = [0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.9]
qs_neg = [0.9, 0.95, 0.99, 0.999]

print("\nScore quantiles:")
print(quantiles(df["pred"], qs_all))

print("\nPositive score quantiles:")
print(
    quantiles(
        df.filter(pl.col("label") == 1)["pred"],
        qs_pos
    )
)

print("\nNegative score quantiles:")
print(
    quantiles(
        df.filter(pl.col("label") == 0)["pred"],
        qs_neg
    )
)


Score quantiles:
{0.5: 0.0022924976158650704, 0.75: 0.008009151826211517, 0.9: 0.03234400385294913, 0.95: 0.32753023883783167, 0.99: 0.9834210558192377, 0.999: 0.9850434641779667}

Positive score quantiles:
{0.01: 0.0262724088771964, 0.05: 0.064502863917362, 0.1: 0.6208320364575429, 0.25: 0.9420871866847204, 0.5: 0.976343505319006, 0.75: 0.98354351230848, 0.9: 0.9850434641779667}

Negative score quantiles:
{0.9: 0.021487541739773013, 0.95: 0.040264979931965386, 0.99: 0.5997568930876386, 0.999: 0.9661084294552754}


In [35]:
import polars as pl
import numpy as np

df = pl.read_parquet("work/task04/val_predictions.parquet")

TH = 0.926

# Positive/negative rows at threshold
pos = df.filter(pl.col("label") == 1)
neg = df.filter(pl.col("label") == 0)

print("=== PAIR COUNTS ===")
print("Total rows:", df.height)
print("True candidate pairs:", pos.height)
print("False candidate pairs:", neg.height)

print("\nAt threshold =", TH)
print("TP:", pos.filter(pl.col("pred") >= TH).height)
print("FN:", pos.filter(pl.col("pred") < TH).height)
print("FP:", neg.filter(pl.col("pred") >= TH).height)
print("TN:", neg.filter(pl.col("pred") < TH).height)

print("\n=== POSITIVE RECALL BY SCORE BAND ===")
for lo, hi in [
    (0.0, 0.5),
    (0.5, 0.7),
    (0.7, 0.8),
    (0.8, 0.9),
    (0.9, 0.926),
    (0.926, 0.95),
    (0.95, 0.98),
    (0.98, 1.0),
]:
    n = pos.filter(
        (pl.col("pred") >= lo) &
        (pl.col("pred") < hi)
    ).height
    print(f"{lo:.3f}-{hi:.3f}: {n}")

print("\n=== HARD NEGATIVES ===")
for th in [0.5, 0.7, 0.8, 0.9, 0.926, 0.95, 0.97, 0.98]:
    n = neg.filter(pl.col("pred") >= th).height
    print(f"negative >= {th:.3f}: {n}")

# Per-S1 selection behaviour
s = (
    df.group_by("source1_entity_id")
      .agg([
          pl.col("label").sum().alias("true_retrieved"),
          (pl.col("pred") >= TH).sum().alias("predicted"),
          ((pl.col("label") == 1) &
           (pl.col("pred") >= TH)).sum().alias("tp"),
          pl.col("pred").max().alias("max_score"),
      ])
)

print("\n=== PER-S1 COUNT ERRORS ===")
print("Exact predicted count:", 
      s.filter(pl.col("predicted") == pl.col("true_retrieved")).height)

print("Under-predicted:",
      s.filter(pl.col("predicted") < pl.col("true_retrieved")).height)

print("Over-predicted:",
      s.filter(pl.col("predicted") > pl.col("true_retrieved")).height)

print("\nLargest under-predictions:")
print(
    s.filter(pl.col("predicted") < pl.col("true_retrieved"))
     .with_columns(
         (pl.col("true_retrieved") - pl.col("predicted"))
         .alias("missed")
     )
     .sort("missed", descending=True)
     .head(20)
)

print("\nLargest over-predictions:")
print(
    s.filter(pl.col("predicted") > pl.col("true_retrieved"))
     .with_columns(
         (pl.col("predicted") - pl.col("true_retrieved"))
         .alias("extra")
     )
     .sort("extra", descending=True)
     .head(20)
)

=== PAIR COUNTS ===
Total rows: 850622
True candidate pairs: 32524
False candidate pairs: 818098

At threshold = 0.926
TP: 25465
FN: 7059
FP: 2111
TN: 815987

=== POSITIVE RECALL BY SCORE BAND ===
0.000-0.500: 2767
0.500-0.700: 947
0.700-0.800: 862
0.800-0.900: 1572
0.900-0.926: 911
0.926-0.950: 1929
0.950-0.980: 9334
0.980-1.000: 14202

=== HARD NEGATIVES ===
negative >= 0.500: 10034
negative >= 0.700: 6415
negative >= 0.800: 4604
negative >= 0.900: 2748
negative >= 0.926: 2111
negative >= 0.950: 1418
negative >= 0.970: 684
negative >= 0.980: 286

=== PER-S1 COUNT ERRORS ===
Exact predicted count: 4679
Under-predicted: 4193
Over-predicted: 1128

Largest under-predictions:
shape: (20, 6)
┌───────────────────┬────────────────┬───────────┬─────┬───────────┬────────┐
│ source1_entity_id ┆ true_retrieved ┆ predicted ┆ tp  ┆ max_score ┆ missed │
│ ---               ┆ ---            ┆ ---       ┆ --- ┆ ---       ┆ ---    │
│ str               ┆ f32            ┆ u32       ┆ u32 ┆ f64       ┆ 

In [36]:
%cd /kaggle/working/amazon_ml_challenge
!git pull

/kaggle/working/amazon_ml_challenge
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 3.32 KiB | 3.32 MiB/s, done.
From https://github.com/manhar1087/amazon-ml-challenge
   ba4871e..770c195  master     -> origin/master
Updating ba4871e..770c195
Fast-forward
 src/modeling/task05_entity_decoding.py | 262 +++++++++++++++++++++++++++++++++
 1 file changed, 262 insertions(+)
 create mode 100644 src/modeling/task05_entity_decoding.py


In [37]:
!git log -1 --oneline

770c195 (HEAD -> master, origin/master, origin/HEAD) Task 05: Add deterministic entity-level decoding


In [38]:
!python src/modeling/task05_entity_decoding.py

Loading Task 04 validation IDs...
Total Validation S1s from Task 04: 10000
Tuning S1s: 5000
Evaluation S1s: 5000
Loading ground truth...
Loading Task 04 predictions...

A. Evaluating Global Threshold Baseline...

B. Evaluating Fixed Top-K...

C. Evaluating Threshold + Top-K Grid...

D. Evaluating Adaptive Cardinality...

BEST POLICY ON TUNING SET:
{
  "name": "Adaptive abs=0.926 rel=0.95",
  "policy": {
    "type": "adaptive",
    "abs_th": 0.926,
    "rel_th": 0.95
  },
  "metrics": {
    "macro_f05": 0.8338385001954332,
    "zero_match_f05": 0.7107142857142857,
    "singleton_f05": 0.6705085857521026,
    "multi_match_f05": 0.8518623295546788,
    "exact_predicted_count": 1994,
    "under_predicted_count": 2580,
    "over_predicted_count": 426,
    "total_predicted_pairs": 13606,
    "total_TP": 12629,
    "total_FP": 977,
    "total_FN": 4725
  }
}

Applying best policy to Untouched Evaluation Set...
FINAL EVALUATION METRICS:
{
  "macro_f05": 0.8319831993148942,
  "zero_match_f05": 

In [39]:
%cd /kaggle/working/amazon_ml_challenge
!git pull
!git log -1 --oneline

/kaggle/working/amazon_ml_challenge
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 8 (delta 2), reused 8 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 6.36 KiB | 3.18 MiB/s, done.
From https://github.com/manhar1087/amazon-ml-challenge
   770c195..bc444f0  master     -> origin/master
Updating 770c195..bc444f0
Fast-forward
 src/features/feature_engineering_v2.py   | 120 +++++++++++++++
 src/modeling/task06_advanced_matching.py | 250 +++++++++++++++++++++++++++++++
 src/modeling/task06_error_analysis.py    |  96 ++++++++++++
 3 files changed, 466 insertions(+)
 create mode 100644 src/features/feature_engineering_v2.py
 create mode 100644 src/modeling/task06_advanced_matching.py
 create mode 100644 src/modeling/task06_error_analysis.py
bc444f0 (HEAD -> master, origin/master, origin/HEAD) Task 06: Advanced matching and ranking


In [40]:
%cd /kaggle/working/amazon_ml_challenge
!PYTHONPATH=/kaggle/working/amazon_ml_challenge python src/modeling/task06_error_analysis.py

/kaggle/working/amazon_ml_challenge
Loading predictions...
Loading data...
Joining features for error analysis...
Error analysis completed and saved to work/task06/error_analysis.json


In [41]:
import lightgbm as lgb
import numpy as np
import torch
import subprocess

print("=== GPU HARDWARE ===")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], 
                     capture_output=True, text=True).stdout)

print("=== PYTORCH ===")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

print("\n=== LIGHTGBM ===")
print("Version:", lgb.__version__)
print("Path:", lgb.__file__)

print("\n=== LIGHTGBM CUDA TEST ===")

X = np.random.rand(10000, 20).astype(np.float32)
y = np.random.randint(0, 2, 10000)

try:
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=20,
        learning_rate=0.1,
        num_leaves=31,
        device_type="cuda",
        verbosity=1
    )

    model.fit(X, y)

    print("\n✅ LIGHTGBM CUDA TEST PASSED")
    print("LightGBM successfully trained using CUDA.")

except Exception as e:
    print("\n❌ LIGHTGBM CUDA TEST FAILED")
    print(type(e).__name__ + ":", e)

=== GPU HARDWARE ===
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

=== PYTORCH ===
CUDA available: True
GPU count: 2

=== LIGHTGBM ===
Version: 4.6.0
Path: /usr/local/lib/python3.12/dist-packages/lightgbm/__init__.py

=== LIGHTGBM CUDA TEST ===
[LightGBM] [Warning] Using sparse features with CUDA is currently not supported.
[LightGBM] [Info] Number of positive: 4955, number of negative: 5045

❌ LIGHTGBM CUDA TEST FAILED
LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1


[LightGBM] [Fatal] CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1


In [42]:
%cd /kaggle/working/amazon_ml_challenge

!PYTHONPATH=/kaggle/working/amazon_ml_challenge \
python src/modeling/task06_advanced_matching.py

/kaggle/working/amazon_ml_challenge
Loading data...
Loading Task 04 splits...
Loading existing Task 04 candidate artifact...
Preparing Train set...
Preparing Validation set...
Building Features V2 (Train)...
Built train features in 8.30s
Building Features V2 (Val)...
Training Binary LightGBM...
Training until validation scores don't improve for 20 rounds
[20]	training's auc: 0.997836	training's average_precision: 0.986382	valid_1's auc: 0.9963	valid_1's average_precision: 0.938477
[40]	training's auc: 0.998717	training's average_precision: 0.990671	valid_1's auc: 0.997142	valid_1's average_precision: 0.95165
[60]	training's auc: 0.99901	training's average_precision: 0.992795	valid_1's auc: 0.997195	valid_1's average_precision: 0.955105
Early stopping, best iteration is:
[52]	training's auc: 0.998901	training's average_precision: 0.991933	valid_1's auc: 0.997337	valid_1's average_precision: 0.955136
Training LambdaRank LightGBM...
Training until validation scores don't improve for 20 ro

In [43]:
%cd /kaggle/working/amazon_ml_challenge
!git pull

/kaggle/working/amazon_ml_challenge
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 4), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 4.60 KiB | 1.53 MiB/s, done.
From https://github.com/manhar1087/amazon-ml-challenge
   bc444f0..cea52bb  master     -> origin/master
Updating bc444f0..cea52bb
Fast-forward
 src/features/feature_engineering_v3.py         | 149 +++++++++++++++++++
 src/modeling/task06_2_targeted_optimization.py | 192 +++++++++++++++++++++++++
 2 files changed, 341 insertions(+)
 create mode 100644 src/features/feature_engineering_v3.py
 create mode 100644 src/modeling/task06_2_targeted_optimization.py


In [44]:
!PYTHONPATH=/kaggle/working/amazon_ml_challenge python src/modeling/task06_2_targeted_optimization.py

Loading data...
Loading Task 04 splits...
Loading existing Task 04 candidate artifact...
Preparing Train set...
Preparing Validation set...
Building Features V2...
Building Features V3...

Training Model: Binary_V2
Training until validation scores don't improve for 30 rounds
[20]	training's auc: 0.997836	training's average_precision: 0.986382	valid_1's auc: 0.9963	valid_1's average_precision: 0.938477
[40]	training's auc: 0.998717	training's average_precision: 0.990671	valid_1's auc: 0.997142	valid_1's average_precision: 0.95165
[60]	training's auc: 0.99901	training's average_precision: 0.992795	valid_1's auc: 0.997195	valid_1's average_precision: 0.955105
[80]	training's auc: 0.999211	training's average_precision: 0.994396	valid_1's auc: 0.997319	valid_1's average_precision: 0.95743
[100]	training's auc: 0.999329	training's average_precision: 0.995222	valid_1's auc: 0.997416	valid_1's average_precision: 0.958459
[120]	training's auc: 0.999422	training's average_precision: 0.995894	val

In [45]:
%cd /kaggle/working/amazon_ml_challenge
!git pull
!git log -1 --oneline

/kaggle/working/amazon_ml_challenge
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 4.64 KiB | 1.55 MiB/s, done.
From https://github.com/manhar1087/amazon-ml-challenge
   cea52bb..22fc866  master     -> origin/master
Updating cea52bb..22fc866
Fast-forward
 src/features/feature_engineering_v4.py         | 159 +++++++++++++++++++
 src/modeling/task06_2_targeted_optimization.py |  21 ++-
 src/modeling/task06_3_tune_slicer.py           | 105 +++++++++++++
 src/modeling/task06_4_v3_vs_v4.py              | 207 +++++++++++++++++++++++++
 4 files changed, 485 insertions(+), 7 deletions(-)
 create mode 100644 src/features/feature_engineering_v4.py
 create mode 100644 src/modeling/task06_3_tune_slicer.py
 create mode 100644 src/modeling/task06_4_v3_vs_v4.py
22fc866 (HEAD -> master, origin/master, origin/HEAD) Task

In [46]:
!PYTHONPATH=/kaggle/working/amazon_ml_challenge python src/modeling/task06_4_v3_vs_v4.py

Loading data...
Loading Task 04 splits...
Loading existing Task 04 candidate artifact...
Preparing Train set...
Preparing Validation set...
Building Features V3...
Building Features V4...

Training Model: Binary_V3
Training until validation scores don't improve for 30 rounds
[20]	training's auc: 0.998562	training's average_precision: 0.99091	valid_1's auc: 0.997262	valid_1's average_precision: 0.955948
[40]	training's auc: 0.999169	training's average_precision: 0.99428	valid_1's auc: 0.99795	valid_1's average_precision: 0.966153
[60]	training's auc: 0.9994	training's average_precision: 0.995873	valid_1's auc: 0.998035	valid_1's average_precision: 0.968305
[80]	training's auc: 0.999531	training's average_precision: 0.996741	valid_1's auc: 0.998246	valid_1's average_precision: 0.971041
[100]	training's auc: 0.999617	training's average_precision: 0.997278	valid_1's auc: 0.998325	valid_1's average_precision: 0.972266
[120]	training's auc: 0.99968	training's average_precision: 0.997744	vali

In [47]:
%cd /kaggle/working/amazon_ml_challenge
!git pull
!git log -1 --oneline

/kaggle/working/amazon_ml_challenge
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 5.37 KiB | 2.68 MiB/s, done.
From https://github.com/manhar1087/amazon-ml-challenge
   22fc866..560f609  master     -> origin/master
Updating 22fc866..560f609
Fast-forward
 src/modeling/task06_5_candidate_optimization.py | 409 ++++++++++++++++++++++++
 1 file changed, 409 insertions(+)
 create mode 100644 src/modeling/task06_5_candidate_optimization.py
560f609 (HEAD -> master, origin/master, origin/HEAD) Task 06.5: Optimize candidate recall


In [48]:
!PYTHONPATH=/kaggle/working/amazon_ml_challenge \
python src/modeling/task06_5_candidate_optimization.py

Loading Task 04 splits...
Loading Ground Truth...
/kaggle/working/amazon_ml_challenge/src/modeling/task06_5_candidate_optimization.py:196: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  gt_df = gt_df.with_columns(pl.col("matched_entity_ids").str.split(",").alias("candidate_entity_id")).explode("candidate_entity_id")
Traceback (most recent call last):
  File "/kaggle/working/amazon_ml_challenge/src/modeling/task06_5_candidate_optimization.py", line 409, in <module>
    run()
  File "/kaggle/working/amazon_ml_challenge/src/modeling/task06_5_candidate_optimization.py", line 218, in run
    with open("retrieval_setup/tfidf_name/corpus_eids.pkl", "rb") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'retrieval_setup/tfidf_name/corpus_eids.pkl'
